In [ ]:
from pathlib import Path
import csv
import json
import math
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

In [ ]:
# Use relative paths so the notebook can be moved to another machine.
PROBABILITY_INPUT_DIR = Path('outputs/base_probabilities')
FUSION_OUTPUT_DIR = Path('outputs/fusion')
REGULARIZATION_C = 0.01
MAX_ITER = 5000
RANDOM_STATE = 42

In [ ]:
MODALITIES = ('rgb', 'tir', 'dem')
MODALITY_PAIRS = (('rgb', 'tir'), ('rgb', 'dem'), ('tir', 'dem'))
CLASS_NAMES = (
    'Background',
    'Debris Accumulation',
    'Exposed Rock Mass',
    'Agricultural Encroachment',
    'Drainage Infrastructure',
)
PROBABILITY_CLIP = 1e-6
META_FEATURE_DIMENSION = 36

def clipped_log_odds(probabilities):
    p = np.clip(probabilities, PROBABILITY_CLIP, 1.0 - PROBABILITY_CLIP)
    return np.log(p / (1.0 - p))

def prediction_margin(probabilities):
    ordered = np.sort(probabilities, axis=1)
    return (ordered[:, -1] - ordered[:, -2]).reshape(-1, 1)

def disagreement(first, second):
    return np.mean(np.abs(first - second), axis=1).reshape(-1, 1)

def class_weights_sqrt(labels):
    counts = np.bincount(labels, minlength=len(CLASS_NAMES))
    total = len(labels)
    if np.any(counts == 0):
        raise ValueError(f'Missing class in the training fold: {counts.tolist()}')
    return {class_id: math.sqrt(total / (len(CLASS_NAMES) * count)) for class_id, count in enumerate(counts)}

def build_meta_features(probabilities):
    parts = []
    names = []
    for modality in MODALITIES:
        parts.append(probabilities[modality])
        names.extend(f'{modality}_probability_class_{class_id}' for class_id in range(len(CLASS_NAMES)))
    for modality in MODALITIES:
        parts.append(clipped_log_odds(probabilities[modality]))
        names.extend(f'{modality}_log_odds_class_{class_id}' for class_id in range(len(CLASS_NAMES)))
    for modality in MODALITIES:
        parts.append(prediction_margin(probabilities[modality]))
        names.append(f'{modality}_margin')
    for first, second in MODALITY_PAIRS:
        parts.append(disagreement(probabilities[first], probabilities[second]))
        names.append(f'{first}_{second}_mean_absolute_probability_difference')
    meta_features = np.concatenate(parts, axis=1)
    if meta_features.shape[1] != META_FEATURE_DIMENSION:
        raise ValueError(f'Expected 36 meta-features, got {meta_features.shape[1]}')
    return meta_features, names

def check_probabilities(probabilities, name):
    if probabilities.ndim != 2 or probabilities.shape[1] != len(CLASS_NAMES):
        raise ValueError(f'{name} has an invalid shape: {probabilities.shape}')
    if np.any(probabilities < -1e-8) or np.any(probabilities > 1.0 + 1e-8):
        raise ValueError(f'{name} contains values outside [0, 1].')
    if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f'{name} must contain finite probability rows that sum to one.')

In [ ]:
fold_paths = sorted(PROBABILITY_INPUT_DIR.glob('fold_*_probabilities.npz'))
if len(fold_paths) != 5:
    raise ValueError(f'Expected five probability files, found {len(fold_paths)}.')
FUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR = FUSION_OUTPUT_DIR / 'models'
MODEL_OUTPUT_DIR.mkdir(exist_ok=True)
metric_rows = []

for fold_number, fold_path in enumerate(fold_paths, start=1):
    with np.load(fold_path, allow_pickle=False) as archive:
        train_ids = archive['train_sample_ids'].astype(str)
        test_ids = archive['test_sample_ids'].astype(str)
        y_train = archive['y_train'].astype(np.int64)
        y_test = archive['y_test'].astype(np.int64)
        train_probabilities = {modality: archive[f'{modality}_train_probabilities'].astype(float) for modality in MODALITIES}
        test_probabilities = {modality: archive[f'{modality}_test_probabilities'].astype(float) for modality in MODALITIES}

    for modality in MODALITIES:
        check_probabilities(train_probabilities[modality], f'{modality} train fold {fold_number}')
        check_probabilities(test_probabilities[modality], f'{modality} test fold {fold_number}')
    train_meta, feature_names = build_meta_features(train_probabilities)
    test_meta, _ = build_meta_features(test_probabilities)

    scaler = StandardScaler()
    train_meta_scaled = scaler.fit_transform(train_meta)
    test_meta_scaled = scaler.transform(test_meta)
    meta_learner = LogisticRegression(
        C=REGULARIZATION_C,
        penalty='l2',
        solver='lbfgs',
        class_weight=class_weights_sqrt(y_train),
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
    )
    meta_learner.fit(train_meta_scaled, y_train)
    fused_probabilities = meta_learner.predict_proba(test_meta_scaled)
    predictions = np.argmax(fused_probabilities, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predictions, labels=np.arange(len(CLASS_NAMES)), average='macro', zero_division=0
    )
    metrics = {
        'fold': fold_number,
        'accuracy': float(accuracy_score(y_test, predictions)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
    }
    metric_rows.append(metrics)
    np.savez_compressed(
        FUSION_OUTPUT_DIR / f'fold_{fold_number:02d}_fusion_predictions.npz',
        train_sample_ids=train_ids, test_sample_ids=test_ids, y_test=y_test,
        predictions=predictions.astype(np.int64), fused_probabilities=fused_probabilities,
        meta_feature_names=np.asarray(feature_names, dtype=str),
    )
    joblib.dump({'scaler': scaler, 'classifier': meta_learner, 'meta_feature_names': feature_names}, MODEL_OUTPUT_DIR / f'fold_{fold_number:02d}_fusion_model.joblib')
    print(f'Fold {fold_number}: macro-F1={f1:.4f}')

In [ ]:
metric_names = ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1')
summary = {
    metric: {'mean': float(np.mean([row[metric] for row in metric_rows])), 'sd': float(np.std([row[metric] for row in metric_rows], ddof=1))}
    for metric in metric_names
}
report = {
    'folds': metric_rows,
    'summary': summary,
    'configuration': {
        'class_names': list(CLASS_NAMES),
        'meta_feature_dimension': META_FEATURE_DIMENSION,
        'probability_clip': PROBABILITY_CLIP,
        'standardization': 'fit on outer-training meta-features only',
        'class_weights': 'square root of inverse-frequency weights',
        'logistic_regression': {'C': REGULARIZATION_C, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': MAX_ITER},
    },
}
(FUSION_OUTPUT_DIR / 'fusion_metrics.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
with (FUSION_OUTPUT_DIR / 'fusion_fold_metrics.csv').open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=('fold', *metric_names))
    writer.writeheader()
    writer.writerows(metric_rows)
for metric in metric_names:
    print(f'{metric}: {summary[metric]["mean"]:.4f} +/- {summary[metric]["sd"]:.4f}')